## LIBRARIES

In [19]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as f, types
import findspark
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
import os
from pathlib import Path
import geopandas as gpd
from shapely import wkt
import json
from aux_functions_wealth import *

## PATHS

In [20]:
projectRoot = Path.cwd().parents[2]
OUTPUT_Y_POIS_DEMAND = str(projectRoot / "data" / "clean" / "wealthAnalysis" /"yellow_zone_demand_pois.parquet")
OUTPUT_FHV_POIS_DEMAND = str(projectRoot / "data" / "clean" / "wealthAnalysis" /"fhv_zone_demand_pois.parquet")
ROLLING_SALES_INPUT = projectRoot / "data" / "rollingprices" / "rollingsales_nyc.csv"
ROLLING_SALES_OUTPUT_Y =  projectRoot / "data" / "clean" / "wealthAnalysis" / "rollingsales_nyc_yellow.csv"
ROLLING_SALES_OUTPUT_FHV =  projectRoot / "data" / "clean" / "wealthAnalysis" / "rollingsales_nyc_fhv.csv"



## SPARK CONFIGURATION

In [21]:
findspark.init()

#Adjust these settings based on your system's resources and requirements


# spark = (SparkSession.builder
#         .master("local[6]")  
#         .config("spark.driver.memory", "8g")
#         .config("spark.executor.memory", "8g")
#         .config("spark.sql.shuffle.partitions", "100")
#         .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "64m")
#         .config("spark.driver.maxResultSize", "1g")
#         .config("spark.sql.session.timeZone", "America/New_York")
#         .config("spark.sql.timestampType", "TIMESTAMP_NTZ")
#         # Network binding fixes for macOS
#         .config("spark.driver.host", "127.0.0.1")
#         .config("spark.driver.bindAddress", "127.0.0.1")
#         .getOrCreate())

spark = (SparkSession.builder
        .master("local[*]")  
        .config("spark.driver.memory", "16g")  
        .config("spark.executor.memory", "16g") 
        .config("spark.memory.fraction", "0.8") 
        .config("spark.sql.shuffle.partitions", "200")
        .config("spark.sql.adaptive.enabled", "true") 
        .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "128m") 
        .config("spark.driver.maxResultSize", "4g") 
        .config("spark.sql.session.timeZone", "America/New_York")
        .config("spark.sql.timestampType", "TIMESTAMP_NTZ")
        .config("spark.driver.host", "127.0.0.1")
        .config("spark.driver.bindAddress", "127.0.0.1")
        .getOrCreate())

In [22]:
rollingsales = spark.read.csv(str(ROLLING_SALES_INPUT), header=True, inferSchema=True)
rollingsales = rollingsales.withColumn("taxi_zone_id", f.col("taxi_zone_id").cast(types.IntegerType()))
rollingsales = rollingsales.filter((f.col("GROSS SQUARE FEET").isNotNull()) & (f.col("GROSS SQUARE FEET") > 0))
rollingsales = rollingsales.filter((f.col("SALE PRICE").isNotNull()) & (f.col("SALE PRICE") > 0))
rollingsales = rollingsales.withColumn("price_per_sqft", f.col("SALE PRICE") / f.col("GROSS SQUARE FEET"))
rollingsales = rollingsales.filter(f.col("price_per_sqft")> 100)

limitq = rollingsales.approxQuantile("price_per_sqft", [0.95], 0.01)[0]
rollingsales = rollingsales.withColumn("price_per_sqft", f.when(f.col("price_per_sqft") > limitq, limitq)
                                       .otherwise(f.col("price_per_sqft")))

rollingsales_zone = rollingsales.groupBy("taxi_zone_id").agg(
    f.mean("price_per_sqft").alias("mean_price_per_sqft"),
    f.percentile_approx("price_per_sqft", 0.5).alias("median_price_per_sqft"),
    f.count(f.col("*")).alias("num_sells")
)
df_rollingsales_zone = rollingsales_zone.toPandas()

## LOAD DEMAND 

In [23]:
df_demand_yellow = pd.read_parquet(str(OUTPUT_Y_POIS_DEMAND))
df_demand_fhv = pd.read_parquet(str(OUTPUT_FHV_POIS_DEMAND))

In [24]:
df_aux_y = df_demand_yellow[["PULocationID","pickup_borough","pickup_zone", 'avg_daily_pickups', 'avg_tip_pct','median_fare', 'trips_to_high_SES_pct','low_SES_density']]
df_rollingsales_zone_merge_y = pd.merge(df_aux_y, df_rollingsales_zone, how="left", right_on="taxi_zone_id", left_on="PULocationID")

In [25]:
df_aux_fhv = df_demand_fhv[["PULocationID","pickup_borough","pickup_zone", 'avg_daily_pickups', 'avg_tip_pct','median_fare', 'trips_to_high_SES_pct','low_SES_density']]
df_rollingsales_zone_merge_fhv = pd.merge(df_aux_fhv, df_rollingsales_zone, how="left", right_on="taxi_zone_id", left_on="PULocationID")

## SAVE DEMAND WIHT HOUSE PRICES

In [26]:
df_rollingsales_zone_merge_y.to_parquet(ROLLING_SALES_OUTPUT_Y, compression = "snappy")
df_rollingsales_zone_merge_fhv.to_parquet(ROLLING_SALES_OUTPUT_FHV, compression = "snappy")

In [27]:
#spark.stop